## WorldView Stereopair Processing: SpaceNet Atlanta Example — 18° convergence

This notebook demonstrates a stereo processing workflow for WorldView-2 satellite imagery using the NASA Ames Stereo Pipeline (ASP) on one of three pairs selected from the [SpaceNet Off-Nadir Building Detection Challenge](https://spacenet.ai/off-nadir-building-detection/) dataset.

Pair `18deg_0d`:

| catid | tile | off-nadir (°) | |
|---|---|---|---|
| `1030010003993E00` | P001 | 11.2 | left |
| `1030010003CAF100` | P002 | 10.6 | right |

Pair geometry (from `StereopairMetadataParser`):

- **Convergence angle**: 18.07°
- **Base-to-height ratio**: 0.32
- **BIE angle**: 81.69° (ideal 90°)
- **Asymmetry angle**: 0.10° (ideal 0°)
- **Temporal separation**: 0 days (same-pass MVS collect, 2009-12-22)
- **Intersection overlap**: 88.9%

### Why this pair?

All 27 WV2 CATIDs in this dataset were acquired on a **single orbit pass** (2009-12-22), making this an MVS collection with zero temporal separation. We target **~15–25° convergence** to balance occlusion on built-up structures against height precision over flatter terrain. See the [scene-selection notebook](./worldview_spacenet_atlanta_stereo_scene_selection.ipynb) for the full pair-ranking analysis.

The pipeline below (wv_correct → bundle_adjust → mapproject → parallel_stereo → point2dem → geodiff) is shared across all three pair notebooks and runs on the **same Atlanta airport ROI**, so the resulting DEMs are directly comparable.

---

### Processing Overview

- **Data Retrieval** — Download WorldView-2 tiles from AWS S3
- **CCD Artifact Correction** — Apply `wv_correct` to fix WV2 sensor artifacts
- **Reference DEM Preparation** — Copernicus 30m DEM with WGS84 ellipsoid heights
- **Bundle Adjustment** — Camera model refinement on the full tile
- **Mapprojection** — Project images onto reference DEM over the Atlanta airport ROI
- **Stereo Processing** — Generate DEM from the stereopair
- **Visualization with `asp_plot`** — Plots, ICESat-2 comparison, PDF report


---

## Data Retrieval

The full catalogue of WV2 CATIDs and the pair-selection analysis is documented in the companion notebook [`worldview_spacenet_atlanta_stereo_scene_selection.ipynb`](./worldview_spacenet_atlanta_stereo_scene_selection.ipynb). This notebook processes the **`18deg_0d`** pair chosen there.

### Download Commands

The SpaceNet Atlanta raw L1B images are in `s3://spacenet-dataset/AOIs/AOI_6_Atlanta/metadata/<CATID>/<workorder>_<PXXX>_PAN/`. Download only the specific tiles needed for this pair:

```bash
$ mkdir -p atlanta_stereo_18deg_0d/images
$ cd atlanta_stereo_18deg_0d/images

# Download left image tile: 1030010003993E00 P001
$ aws s3 --no-sign-request sync \
  s3://spacenet-dataset/AOIs/AOI_6_Atlanta/metadata/1030010003993E00/ \
  aws/1030010003993E00/ \
  --exclude "*" --include "*P001_PAN/*"

# Download right image tile: 1030010003CAF100 P002
$ aws s3 --no-sign-request sync \
  s3://spacenet-dataset/AOIs/AOI_6_Atlanta/metadata/1030010003CAF100/ \
  aws/1030010003CAF100/ \
  --exclude "*" --include "*P002_PAN/*"
```

Copy the TIF and XML files to the working directory with short names:

```bash
$ cd atlanta_stereo_18deg_0d
$ cp images/aws/1030010003993E00/*P001_PAN/*P1BS*P001.TIF 1030010003993E00_P001.tif
$ cp images/aws/1030010003993E00/*P001_PAN/*P1BS*P001.XML 1030010003993E00_P001.xml
$ cp images/aws/1030010003CAF100/*P002_PAN/*P1BS*P002.TIF 1030010003CAF100_P002.tif
$ cp images/aws/1030010003CAF100/*P002_PAN/*P1BS*P002.XML 1030010003CAF100_P002.xml
```


### Note on Tiling

Each Atlanta CATID is delivered as multiple tiles (`P001`, `P002`, `P003`). For this pair we use individual tiles (`P001` from `1030010003993E00` and `P002` from `1030010003CAF100`) that each fully contain the airport ROI. Mosaicking with `dg_mosaic` is not needed.


### Stereo Geometry Analysis

Before processing, it's useful to analyze the stereo acquisition geometry to verify the convergence angle and other geometric properties. We use `asp_plot` to visualize the geometry from the XML camera metadata:

In [ ]:
# Set the base directory for your processing
directory = "~/Desktop/asp-plot-examples/atlanta_stereo_18deg_0d/"

from asp_plot.stereo_geometry import StereoGeometryPlotter

sgp = StereoGeometryPlotter(directory)
utm_code = sgp.get_pair_utm_epsg()
map_crs = f"EPSG:{utm_code}"
scene_bounds = sgp.get_scene_bounds()

print(f"Auto-detected UTM projection: {map_crs}")
print(f"Scene extent (lon/lat): {scene_bounds[0]:.6f} {scene_bounds[1]:.6f} {scene_bounds[2]:.6f} {scene_bounds[3]:.6f}")

sgp.dg_geom_plot()


### Satellite Position and Orientation Data

The XML camera metadata also contains ephemeris (position/velocity) and attitude (orientation quaternion) data reported by the satellite during acquisition. Visualizing this data can help assess the quality of the raw metadata before ASP processing.

In [ ]:
sgp.satellite_position_orientation_plot()

In the above plot:

- Top row (Position Covariance): A map showing the satellite's path over the ground during image capture, colored by how uncertain the satellite's reported position is (in meters). Lower values mean the satellite knows where it is more precisely.

- Middle row (Roll / Pitch / Yaw): Shows how the satellite's pointing deviates from its expected nadir-pointing orientation over time. Roll is rotation around the flight direction, pitch is rotation around the cross-track axis, and yaw is rotation around the nadir axis. Values near zero mean the satellite is pointing straight down. These are computed by comparing the raw attitude quaternions against a reference orientation estimated from the satellite's orbital position and velocity.

- Bottom row (Attitude Covariance Trace): Shows how uncertain the satellite's orientation knowledge is over time. Lower values mean the satellite is more confident about which direction it's pointing. Spikes or jumps indicate moments of less reliable pointing knowledge, which could affect image quality in those portions of the scene.

---

## CCD Artifact Correction

WorldView-2 imagery has subpixel CCD boundary misalignments. The SpaceNet Atlanta data was processed in 2019 (before the May 2022 improvement), so `wv_correct` is required:

```bash
$ cd atlanta_stereo_18deg_0d

$ wv_correct 1030010003993E00_P001.tif 1030010003993E00_P001.xml 1030010003993E00_P001_corr.tif
$ wv_correct 1030010003CAF100_P002.tif 1030010003CAF100_P002.xml 1030010003CAF100_P002_corr.tif
```

The `*_corr.tif` corrected images are used for all subsequent processing steps.


---

## Processing Configuration

```bash
$ cd atlanta_stereo_18deg_0d

# Target spatial reference system (UTM Zone 16N for Atlanta)
$ t_srs="EPSG:32616"

# Processing ROI: Atlanta airport area (~5.7 x 5.4 km)
$ t_projwin="735685 3722295 741350 3727695"

# Output resolution — use the more nadir image's meanProductGSD
$ tr=0.481

# Reference DEM filename (WGS84 ellipsoid heights, projected to UTM)
$ reference_dem_fn="ref/cop30_atlanta_wgs84_utm.tif"
```


---

## Reference DEM Preparation

A reference DEM is required for mapprojecting the input images and validating the output DEM. We use the Copernicus 30m GLO-30 DEM.

### Download Copernicus DEM with Ellipsoid Heights

```bash
$ cd atlanta_stereo_18deg_0d

$ python /path/to/your/fetch_dem/download_global_DEM.py \
  -demtype COP30_E \
  -extent "$scene_bounds" \
  -out_fn "$reference_dem_fn" \
  -out_proj "$t_srs" \
  -apikey [YOUR_OPEN_TOPOGRAPHY_API_KEY]
```


---

## Bundle Adjustment

Run on the **full tile** (no ROI cropping):

```bash
$ cd atlanta_stereo_18deg_0d

$ bundle_adjust \
  --threads 8 \
  --ip-per-image 10000 \
  --tri-weight 0.1 \
  --tri-robust-threshold 0.1 \
  --camera-weight 0 \
  1030010003993E00_P001_corr.tif 1030010003CAF100_P002_corr.tif \
  1030010003993E00_P001.xml 1030010003CAF100_P002.xml \
  -o ba/run
```


### Bundle Adjustment Results

Visualize the bundle adjustment results to verify the optimization reduced reprojection errors:

In [ ]:
from asp_plot.bundle_adjust import ReadBundleAdjustFiles, PlotBundleAdjustFiles
import contextily as ctx

# Define subdirectories
bundle_adjust_directory = "ba/"
stereo_directory = "stereo/"

# Map configuration (map_crs was auto-detected in the geometry cell above)
ctx_kwargs = {
    "crs": map_crs,
    "source": ctx.providers.Esri.WorldImagery,
    "attribution_size": 0,
    "alpha": 0.5,
}

# Read bundle adjustment residuals
ba_files = ReadBundleAdjustFiles(directory, bundle_adjust_directory)
resid_initial_gdf, resid_final_gdf = ba_files.get_initial_final_residuals_gdfs(residuals_in_meters=True)

# Plot residuals before and after optimization
plotter = PlotBundleAdjustFiles(
    [resid_initial_gdf, resid_final_gdf],
    lognorm=True,
    title="Bundle Adjust Initial and Final Residuals"
)
plotter.plot_n_gdfs(
    column_name="mean_residual",
    cbar_label="Mean residual (px)",
    map_crs=map_crs,
    **ctx_kwargs
)

---

## Define Region of Interest

The Atlanta airport ROI (~5.7 × 5.4 km) is shared across all three pair notebooks. The [scene-selection notebook](./worldview_spacenet_atlanta_stereo_scene_selection.ipynb) verified that this ROI is fully contained in both tiles.

```bash
# ROI in UTM Zone 16N (EPSG:32616): xmin ymin xmax ymax
$ t_projwin="735685 3722295 741350 3727695"
```


In [ ]:
# Get the intersection extent in UTM coordinates (from the geometry cell above)
intersection_bounds = sgp.get_intersection_bounds(epsg=utm_code)
t_projwin = f"{intersection_bounds[0]:.0f} {intersection_bounds[1]:.0f} {intersection_bounds[2]:.0f} {intersection_bounds[3]:.0f}"
print(f"Intersection projwin: {t_projwin}")

As noted above, we use the shared ROI `t_projwin="735685 3722295 741350 3727695"` — the Atlanta airport area.


---

## Mapprojection

```bash
$ cd atlanta_stereo_18deg_0d

$ mapproject \
  -t rpc --processes 4 --threads 2 \
  --tr $tr --t_projwin $t_projwin --t_srs "$t_srs" \
  --bundle-adjust-prefix ba/run \
  "$reference_dem_fn" \
  1030010003993E00_P001_corr.tif 1030010003993E00_P001.xml \
  1030010003993E00_P001_corr_map.tif

$ mapproject \
  -t rpc --processes 4 --threads 2 \
  --tr $tr --t_projwin $t_projwin --t_srs "$t_srs" \
  --bundle-adjust-prefix ba/run \
  "$reference_dem_fn" \
  1030010003CAF100_P002_corr.tif 1030010003CAF100_P002.xml \
  1030010003CAF100_P002_corr_map.tif
```


---

## Stereo Processing

```bash
$ cd atlanta_stereo_18deg_0d

$ parallel_stereo \
  --stereo-algorithm asp_mgm --subpixel-mode 9 \
  --processes 2 --threads 4 --alignment-method none \
  --bundle-adjust-prefix ba/run \
  1030010003993E00_P001_corr_map.tif 1030010003CAF100_P002_corr_map.tif \
  1030010003993E00_P001.xml 1030010003CAF100_P002.xml \
  stereo/run "$reference_dem_fn"

# Generate ~1.9 m DEM (~4x the native input GSD)
$ point2dem --tr 1.9 --t_srs "$t_srs" --errorimage stereo/run-PC.tif

# Difference map vs. reference DEM
$ geodiff stereo/run-DEM.tif "$reference_dem_fn" -o stereo/run_vs_ref
```


### Stereo Results Visualization

Examine the stereo processing outputs to assess DEM quality:

In [ ]:
from asp_plot.stereo import StereoPlotter
from asp_plot.scenes import ScenePlotter

# Plot input scenes (mapprojected images)
scene_plotter = ScenePlotter(directory, stereo_directory, title="Mapprojected Input Scenes")
scene_plotter.plot_scenes()

# Plot DEM results
stereo_plotter = StereoPlotter(directory, stereo_directory)
stereo_plotter.title = "Stereo DEM Results"
stereo_plotter.plot_dem_results()

# Plot detailed hillshade with zoom subsets
stereo_plotter.title = "Hillshade with Details"
stereo_plotter.plot_detailed_hillshade(subset_km=0.5)

---

## Comprehensive Report and ICESat-2 Altimetry Validation

In addition to the inline visualizations above, we can validate the ASP DEM against ICESat-2 ATL06-SR altimetry data. This provides an independent accuracy assessment using satellite laser altimetry.

The sections below demonstrate ICESat-2 comparison and generate a comprehensive PDF report.

### Setup

After you have `asp_plot` installed and ready to use, you can set up the directory and file names:

In [ ]:
# Set the base directory for your processing
directory = "~/Desktop/asp-plot-examples/atlanta_stereo_18deg_0d/"

bundle_adjust_directory = "ba/"
stereo_directory = "stereo/"


### Full Report Generation

Generate a comprehensive PDF report:

#### **Note:** The generated report is saved to `reports/WorldView_Atlanta_18deg_0d-asp-plot-report.pdf`.


In [ ]:
!asp_plot \
  --directory $directory \
  --bundle_adjust_directory $bundle_adjust_directory \
  --stereo_directory $stereo_directory \
  --subset_km 0.5 \
  --report_filename ../../reports/WorldView_Atlanta_18deg_0d-asp-plot-report.pdf


### ICESat-2 Altimetry Validation

Compare the ASP DEM with ICESat-2 ATL06-SR altimetry data:

In [ ]:
from asp_plot.altimetry import Altimetry

In [ ]:
icesat = Altimetry(
  directory=directory,
  dem_fn=stereo_plotter.dem_fn
)

In [ ]:
# Request ATL06-SR data (single "all" processing level)
icesat.request_atl06sr_multi_processing(
    processing_levels=["all"],
    save_to_parquet=True,
)

In [ ]:
# Filter out water bodies
icesat.filter_esa_worldcover(filter_out="water")

No temporal filtering needed for the simplified report workflow.
Advanced temporal filtering is still available via:

```py
icesat.predefined_temporal_filter_atl06sr(date=...)
icesat.generic_temporal_filter_atl06sr(select_years=..., select_months=..., ...)
```

In [ ]:
# Map view of ICESat-2 vs DEM differences
icesat.mapview_plot_atl06sr_to_dem(
    key="all",
    map_crs=map_crs,
    **ctx_kwargs,
)

In [ ]:
# Histogram with per-landcover-class statistics
icesat.histogram_by_landcover(key="all")

In [ ]:
# Profile along the best ICESat-2 track
icesat.plot_atl06sr_dem_profile(key="all")
icesat.plot_best_worst_segments(key="all")

### DEM-to-Altimetry Alignment

Run `pc_align` to compute the translation required to align the DEM with ICESat-2 data:

**Note on multiple pc_align runs:** The `alignment_report` method runs `pc_align` once for each temporal filter (e.g., "ground", "ground_seasonal", etc.). This allows comparison of alignment results across different subsets of ICESat-2 data. Each row in the resulting DataFrame represents a separate alignment run.

The output columns include:
- `north_shift`, `east_shift`, `down_shift`: Translation vector in North-East-Down coordinates (meters)
- `p16_beg/end`, `p50_beg/end`, `p84_beg/end`: Error percentiles before and after alignment

In [ ]:
# Run alignment report to assess DEM quality
icesat.alignment_report(
    processing_level="all",
    minimum_points=100,
    agreement_threshold=0.25,
    write_out_aligned_dem=True,
)

In [ ]:
icesat.alignment_report_df

In [ ]:
# Histogram of aligned DEM vs ICESat-2 differences
icesat.atl06sr_to_dem_dh()

icesat.histogram(
    key="all",
    plot_aligned=True,
)

---

## Next Steps

This notebook processed one of three Atlanta stereo pairs chosen via the [scene-selection notebook](./worldview_spacenet_atlanta_stereo_scene_selection.ipynb). The sibling notebooks process the other two pairs on the same ROI:

- [`worldview_spacenet_atlanta_stereo_16deg_0d.ipynb`](./worldview_spacenet_atlanta_stereo_16deg_0d.ipynb) — 15.97° convergence
- [`worldview_spacenet_atlanta_stereo_18deg_0d.ipynb`](./worldview_spacenet_atlanta_stereo_18deg_0d.ipynb) — 18.07° convergence
- [`worldview_spacenet_atlanta_stereo_22deg_0d.ipynb`](./worldview_spacenet_atlanta_stereo_22deg_0d.ipynb) — 21.77° convergence
